# **ReID**

En este notebook se explica un reentrenamiento del modelo ReID OSNet con imágenes de jugadores de baloncesto, con la hipótesis de que el modelo entrenado específicamente para baloncesto mejorará el seguimiento de los jugadores frente a utilizar un modelo ReID genérico de personas.

A continuación, se explica la reidentificación de forma genérica, seguido del conjunto de datos concreto utilizado, el entrenamiento del modelo, los resultados obtenidos, y limitaciones y posibles líneas a futuro. El código se presenta en otros notebooks de esta misma carpeta. En concreto, en *obtener_datos_reid.ipynb* se encuentra el código utilizado para obtener los datos para entrenar el modelo de reidentificación, y en *entrenamiento_reid.ipynb* el código utilizado para el entrenamiento del modelo y la generación de gráficas.

## **Explicación de ReID**

La reidentificación de personas es un campo ampliamente estudiado, no solo en deportes, sino también en otras áreas como la seguridad. Al inicio de la investigación sobre el tópico, se utilizaron características diseñadas a mano. Sin embargo, con el auge de los modelos de aprendizaje profundo, se han desarrollado modelos de reidentificación basados en esta técnica, que han mejorado la performance de la reidentificación

Para entrenar estos modelos, es necesario contar con datasets con anotaciones de las personas en diferentes escenas y cámaras, de forma que el mismo identificador se asigne a la misma persona. Con estos datasets, es posible entrenar un modelo de reidentificación que, dada una imagen de una persona, proporcione un vector de características. Dadas dos imágenes de personas, la distancia entre sus vectores debe ser menor si son la misma persona, y mayor si son personas diferentes. Este vector se utiliza en algunos algoritmos de seguimiento multi-objeto, junto a métricas de movimiento, para asociar las detecciones de un frame con las identidades seguidas desde frames anteriores.

En la práctica, los conjuntos de datos de reidentificación suelen tener tres particiones. Concretamente, un conjunto de entrenamiento, un conjunto de query y un conjunto de galería. El conjunto de entrenamiento se utiliza para entrenar el modelo, mientras que los dos conjuntos restantes se utilizan para evaluar el modelo, utilizando cada imagen del conjunto de query para buscar una imagen de la misma persona en el conjunto de galería. Para las métricas de evaluación, es común comparar el vector de características de la imagen de query con los de las imágenes de la galería, teniendo en cuenta la misma persona y distintas personas.

Dentro de las técnicas de reidentificación, este proyecto utiliza el modelo OSNet, una red neuronal convolucional profunda que ha demostrado buen rendimiento en tareas de ReID de personas.

OSNet es un modelo de reidentificación diseñado para analizar las imágenes de personas a diferentes escalas. Esto permite que el modelo pueda discriminar entre personas tanto a un nivel más general, como el uniforme del equipo, como a un nivel más detallado, como los rasgos faciales o el pelo.

El modelo extrae características a través de varias ramas de convoluciones, cada una con un tamaño de campo receptivo diferente, para capturar las características de la persona a diferentes escalas. Después, se combinan para obtener el vector final de características con información de todas las escalas. 

Además, la arquitectura de OSNet está diseñada para tener un número bajo de parámetros, lo que la hace menos costosa y menos propensa al sobreajuste durante el entrenamiento, y adecuada para dispositivos con recursos limitados durante la inferencia.

Para el entrenamiento de OSNet, la herramienta torchreid es una librería que permite entrenar modelos de reidentificación de personas, incluyendo OSNet, con datasets propios.

## **Conjunto de datos**

Dado que se trata de una prueba de concepto, se ha realizado el reentrenamiento con un conjunto de datos reducido. Se ha utilizado el conjunto de datos SportsMOT. Dentro de este conjunto de datos, se ha utilizado el conjunto de entrenamiento, dado que el conjunto de validación se utiliza como conjunto de prueba para evaluar el seguimiento en este trabajo.

El conjunto de entrenamiento de SportsMOT contiene secuencias de 3 partidos de baloncesto diferentes. Se han utilizado las secuencias v\_-6Os86HzwCs\_c001 y v\_4LXTUim5anY\_c012 para el conjunto de entrenamiento y la secuencia v\_2j7kLB-vEEk\_c005 para los conjuntos de query y galería, es decir, el conjunto de evaluación. Cada secuencia pertenece a un partido diferente, con jugadores diferentes. Esto permite evaluar el modelo de ReID en un conjunto de datos diferente al del entrenamiento, y comprobar si el modelo generaliza o memoriza las identidades durante el entrenamiento.

Las imágenes de las personas se obtienen de recortar las *bounding boxes* que indica el dataset de seguimiento. No se consideran las imágenes con una altura menor a 48 píxeles y una anchura menor a 24 píxeles, dado que podrían ser demasiado pequeñas para obtener características discriminativas. A su vez, se descartan las imágenes con un IoU mayor a 0.5 con otro *bounding box* de otro jugador, dado que esto podría confundir al modelo al aparecer un jugador parcialmente tapado por otro. Se mantienen los jugadores solapados con IoU menor que 0.5 para que el modelo aprenda a diferenciar jugadores aunque aparezca parte de otro jugador, lo que ocurre con frecuencia en baloncesto.

En ReID entrenado con torchreid, se asume que los conjuntos de query y galería proceden de cámaras distintas, y genera error si se etiquetan con la misma cámara. Dado que sólo se tienen las imágenes de una cámara en el partido, se realiza una división artificial de la secuencia de forma temporal, con la primera mitad de la secuencia usada para el conjunto de query y etiquetada como cámara 0, y la segunda mitad usada para el conjunto de galería y etiquetada como cámara 1. Aunque pertenezcan a la misma cámara, esta se mueve por el campo siguiendo a los jugadores y los jugadores tienen diversas posiciones a lo largo del partido, por lo que hay variación suficiente entre las imágenes para que resulte útil como una aproximación de imágenes de diversas cámaras.

El notebook que realiza esto es `obtener_datos_reid.ipynb`.

## **Entrenamiento del modelo**

Se ha reentrenado el modelo OSNet utilizando el conjunto de entrenamiento durante 12 épocas. Los hiperparámetros del modelo son los parámetros por defecto, dado que son parámetros que suelen funcionar bien de forma general para reidentificación. No se realiza una búsqueda de hiperparámetros dado que el objetivo es realizar una prueba de concepto, y no optimizar el modelo al máximo.

La pérdida utilizada para el entrenamiento se denomina *triplet loss*, una función de pérdida que está diseñada para que, dada un punto ancla $x_a$, un ejemplo de la misma clase $x_p$ (en este caso, el mismo jugador) esté más cerca que un ejemplo de una clase diferente $x_n$ (en este caso, un jugador diferente), por lo menos por un margen $m$. Con este fin, se declara la siguiente función de pérdida:

$$ L_{tri}(\theta) = \sum_{\substack{a, p, n \\ y_a = y_p \neq y_n}} [m + D_{a,p} - D_{a,n}]_+ $$

Siendo $D_{a,p}$ la distancia entre el ejemplo ancla y el de la misma clase (ejemplo positivo), $D_{a,n}$ la distancia entre el ejemplo ancla y el de una clase diferente (ejemplo negativo), y $\theta$ los parámetros del modelo que está siendo entrenado.

Para la elección de los ejemplos positivos y negativos, es necesario escoger ejemplos difíciles, de forma que el modelo aprenda a diferenciar jugadores que se parecen mucho entre sí. Sin embargo, escoger estos ejemplos es computacionalmente costoso, por lo que se opta por dividir el conjunto de entrenamiento en batches con un conjunto aleatorio de $P$ personas y $K$ imágenes por persona. Gracias a esta división, tenemos la siguiente función de pérdida para un batch $X$:

$$ L_{BH}(\theta; X) = \sum_{i=1}^{P} \sum_{a=1}^{K} [m + \max_{p=1...K} D(f_{\theta}(x_a^i), f_{\theta}(x_p^i)) - \min_{\substack{j=1...P \\ \substack{n=1...K \\ j \neq i}}} D(f_{\theta}(x_a^i), f_{\theta}(x_n^j))]_+ $$

Siendo $x_a^i$ el ejemplo ancla, $x_p^i$ un ejemplo positivo de la misma clase, y $x_n^j$ un ejemplo negativo de una clase diferente. Además, $f_{\theta}$ es la función que devuelve el embedding del ejemplo y $D$ la distancia entre los dos embeddings.

Se utiliza un sampler denominado *RandomIdentitySampler*, que escoge de forma aleatoria un número determinado de imágenes por persona, en nuestro caso 4 imágenes por persona. Al tener varias imágenes por persona, se pueden escoger los ejemplos positivos y negativos más difíciles, lo que es importante para *triplet loss*.

En este contexto, Rank-1 y Rank-5 son métricas que miden el porcentaje de veces en las que la identidad correcta aparece en la primera posición o entre las cinco primeras de las imágenes más cercanas en la galería, respectivamente.

Tras el entrenamiento, se debe escoger la época del modelo que se utilizará en el seguimiento. Para ello, se evalúa el modelo en el conjunto de evaluación cada 2 épocas. Al observar las métricas, se aprecia una mejora en Rank-1 y Rank-5 hasta la época 12. También se observa que el modelo encuentra un estancamiento en las métricas de evaluación en Rank-1, y Rank-5 a partir de la época 12, en las que en algunas épocas baja y en otras sube pero no se observa una mejora significativa. La gráfica con la evolución de Rank-1 y Rank-5 se puede observar en la siguiente figura:

![Evolución de Rank-1 y Rank-5 cada dos épocas en el conjunto de evaluación](../img/reid/ev_rank1_rank5_test.png)

*Evolución de Rank-1 y Rank-5 cada dos épocas en el conjunto de evaluación.*

Como el conjunto de evaluación no muestra mejoras consistentes a partir de la época 12, no hay evidencia clara de que el modelo siga aprendiendo a partir de este punto. Se escoge el modelo entrenado hasta esta época para evaluar el seguimiento, con el fin de evitar un posible sobreajuste.

El código utilizado para el entrenamiento del modelo y la generación de gráficas se encuentra en el fichero `entrenamiento_reid.ipynb`.

## **Resultados**

Para probar el modelo, es necesario volver a obtener los parámetros adecuados, dado que si se utilizan los mismos que con el modelo ReID genérico, el seguimiento podría empeorar porque no están optimizados para este nuevo modelo. Como en el algoritmo DeepOCSort sólo se modifica un parámetro, el peso de la apariencia, para optimizarlo, se opta por probar los resultados en este algoritmo.

No se evalúa en el conjunto de entrenamiento de SportsMOT, dado que se ha utilizado para entrenar el modelo, por lo que podría no dar el mejor valor. Es por ello que se prueban los resultados en el conjunto de validación de SportsMOT. Aunque idealmente esto se debería hacer en un conjunto diferente al de entrenamiento y validación para posteriormente evaluar el seguimiento con el hiperparámetro escogido en el conjunto de evaluación, no se dispone de un conjunto de datos diferente, por lo que se opta por utilizar el conjunto de validación.

El mejor valor se obtiene con un peso de 1.0, que consigue un valor de IDF1 de 63.34, en comparación con el valor de 61.95 obtenido con un modelo de ReID genérico, consiguiendo una mejora de 1.39 puntos. 

Esto muestra que un modelo de ReID entrenado específicamente en el contexto del baloncesto puede mejorar el seguimiento de los jugadores. La mejora es pequeña, pero esto era de esperar debido a la gran similitud entre diversos jugadores de baloncesto.

## **Limitaciones y líneas a futuro**

Como limitaciones, el modelo tiene un conjunto de entrenamiento reducido, lo que puede provocar que el modelo no generalice tan bien como modelos entrenados con muchos datos. 

Como líneas a futuro, se plantea ampliar el conjunto de datos de entrenamiento, con el fin de que el modelo pueda generalizar mejor. Además, se podría realizar una búsqueda de hiperparámetros para optimizar el entrenamiento del modelo con el fin de mejorar el modelo resultante en el seguimiento de jugadores de baloncesto.

Por último, las identidades sólo se ven en una cámara, teniendo que simular la existencia de dos cámaras para la evaluación. Podría resultar útil disponer de un conjunto de datos con más cámaras por partido.